# Downscale-Bench meta-evaluation
---

### Introduction
The following Jupyter Notebook is designed to compare different downscaling solution against each other. It provides several plotting routines that allow for a systematic analysis, called meta-evaluation, and enables the user to benchmark a new custom downscaling method against the baselines provided with *DownscaleBench*. <br>
All routines expect that the evaluation steps, the single model evaluation, has been carried out beforehand. Thus, running `main_evaluation.py` as described in the repository's README is a mandatory prerequisite.

### Software requirements
* matplotlib/3.4.3 (version >= 3.4.3)
* numpy (version >= 1.21.3)
* pandas (version >= 2.2.2)
* xarray (version >= 0.20.1)

## **Main**

We start by importing the required Python packages and routines:

In [ ]:
# Import modules
%matplotlib inline
# import os, sys
# base_dir = "../"
# sys.path.append("/p/home/jusers/lehner3/juwels/shared/downscaling_benchmark/packages/evaluation/meta_evaluation")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# from meta_evaluation import load_aggregate_scores

Instead of importing custom modules, we define the needed functions below.

In [ ]:
def visualise_scorecard(scores: pd.DataFrame, variable: str, savepath: str = None):
    metrics = scores.score_name.unique()
    nmetrics = len(metrics)
    fig, axes = plt.subplots(nrows=1, ncols=nmetrics, figsize=(4*nmetrics,4), sharey=True);
    axiter = iter(axes)
    cbar_bounds = [-50, -20, -10, -5, -2, -1, 1, 2, 5, 10, 20, 50]
    col_order = ["YEAR", "DJF", "MAM", "JJA", "SON"]
    for metric, ax in zip(metrics, axiter):
        unit = metric2unit(metric=metric, variable=variable)
        pivot = scores.query(f"score_name == '{metric}'").pivot(columns="time", index="model", values="value")
        pivot = pivot[col_order]
        labeldata = pivot.values
        heatmapdata = (pivot - pivot.loc[config["ref_model"]])/pivot.loc[config["ref_model"]]*100 # *100 to make it percentages
    
        im = heatmap(
            heatmapdata,
            pivot.index.values,
            pivot.columns.values,
            ax=ax,
            title=f"{metric.upper()} [{unit}]",
            cmap="coolwarm",
            norm=matplotlib.colors.BoundaryNorm(boundaries=cbar_bounds, ncolors=256, extend="both"),
            alpha=0.85,
        )
        texts = annotate_heatmap(im, data=labeldata, valfmt="{x:.2f}", threshold=10)
    
    cbar_ax = fig.add_axes([0.3, 0.02, 0.4, 0.04])
    cbar = fig.colorbar(
        im,
        cax=cbar_ax,
        orientation="horizontal",
        extend="both",
        ticks=cbar_bounds,
        label=f"Better <--    % difference vs reference model: {config['ref_model']}    --> Worse",
    )
    plt.suptitle(variable.upper())
    if savepath is None:
        plt.show()
    else:
        plt.savefig(savepath, bbox_inches="tight")
    return None

def metric2unit(metric: str, variable: str):
    if variable == "t2m":
        return {
            "rmse": "K",
            "bias": "K",
            "grad_amplitude": "1",
            "me_std": "K",
            "ralsd": "1",
        }[metric]
    elif variable == "ws100m":
        return {
            "rmse": "m/s",
            "bias": "m/s",
            "grad_amplitude": "1",
            "me_std": "m/s",
            "ralsd": "1",
        }[metric]
    elif variable == "glob_rad":
        return {
            "rmse": "W/m²",
            "bias": "W/m²",
            "grad_amplitude": "1",
            "me_std": "W/m²",
            "ralsd": "1",
            "rmse_relative": "1",
            "bias_relative": "1",
            "fss_thres_50": "1",
            "fss_thres_100": "1",
            "fss_thres_300": "1",
            "fss_thres_500": "1",
        }[metric]

def heatmap(data, row_labels, col_labels, ax=None, title: str = None, **kwargs):
    if not ax:
        ax = plt.gca()
    if not title:
        title = ""
    im = ax.imshow(data, **kwargs);
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=0, ha="center", rotation_mode="anchor")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)
    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)
    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="lightgray", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.set_title(title)
    return im

def annotate_heatmap(im, labels=None, valfmt="{x:.2f}"):
    opts = dict(color="k", ha="center", va="center")
    
    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(labels.shape[0]):
        for j in range(labels.shape[1]):
            text = im.axes.text(j, i, valfmt(labels[i, j], None), **opts);
            texts.append(text)
    return texts

def load_aggregate_scores(config: dict):
    scores = []
    for model in config["models"]:
        scores_file = Path(config["base_folder"], config["variable"], model, "metric_files", "aggregate_scores", "scores.csv")
        scores_iter = pd.read_csv(scores_file, index_col=0)
        scores.append(scores_iter)
    return pd.concat(scores)

### Config parameters
 ---
The following cells rely on an instance of a `Config` class that is used to define parameters for our plotting routines. <br>
The following standard parameters are expected:
 
 - base_folder : define the absolute path location where all the results for all the models are stored e.g. ```"/p/scratch/hclimrep/lehner3/evaluation"```
 - variable : the target downscaling variable e.g. ```"t2m"``` (other options: "ws100m","glob_rad")
 - models : list of competing models e.g. ```["deepru","sha_unet","sha_wgan","swinir"]```. The name of these models should exactly match with their corresponding folder in the base_folder, which is the specific experiment name.

The aggregated scores are then loaded into a `pandas.DataFrame`.

In [ ]:
#config parameters
config = {}
config['base_folder'] = "/p/scratch/hclimrep/lehner3/evaluation"
config['ref_model'] = "Harris WGAN"

In [ ]:
config['variable'] = "t2m"
config['models'] = [
    "deepru_t2m_hres_1",
    "harris_wgan_benchmark_cs256_4gpus_layernorm_added_norm_det",
    "sha_unet_t2m_hres_3",
    "sha_wgan_t2m_hres_3",
    "swinir_latest",
]

In [ ]:
scores_t2m = load_aggregate_scores(config)

In [ ]:
config['variable'] = "ws100m"
config['models'] = [
    "deepru_wind_hres_3",
    "harris_wgan_benchmark_cs256_4gpus_layernorm_recon100_added_norm_det",
    "sha_unet_wind_hres_3",
    "sha_wgan_wind_hres_1",
    "swinir_latest",
]

In [ ]:
scores_ws100m = load_aggregate_scores(config)

In [ ]:
config['variable'] = "glob_rad"
config['models'] = [
    "deepru_rad_hres_1",
    "harris_wgan_benchmark_cs256_4gpus_layernorm_recon100_added_norm_det",
    "sha_unet_rad_hres_1",
    "sha_wgan_rad_hres_3",
    "swinir_latest",
]

In [ ]:
scores_glob_rad = load_aggregate_scores(config)

## Visualize Scorecards
---

Scorecards are visualized per variable, for all metrics. Each metric is displayed in a block, where rows correspond to models and columns to different time aggregations (annual and all seasons).

In [ ]:
visualise_scorecard(scores=scores_t2m, variable="t2m", savepath="scorecard_t2m.png")

In [ ]:
visualise_scorecard(scores=scores_ws100m, variable="ws100m", savepath="scorecard_ws100m.png")

Global Radiance has much more different metrics. the non-standard ones are plotted in a separate figure.

In [ ]:
standard_metrics = ["rmse", "bias", "grad_amplitude", "me_std", "ralsd"]
visualise_scorecard(
    scores=scores_glob_rad.query(f"score_name in {standard_metrics}"),
    variable="glob_rad",
    savepath="scorecard_glob_rad.png"
)

In [ ]:
visualise_scorecard(
    scores=scores_glob_rad.query(f"score_name not in {standard_metrics}"),
    variable="glob_rad",
    savepath="scorecard_glob_rad_extended.png"
)